# 05 — MLflow, Final Prediction & Deployment Handoff

## Rossmann Sales Forecasting — Project 6

This notebook documents the final MLOps stage: MLflow experiment tracking, model artifact logging, MLflow inference, Customer prediction, combined final predictions, validation evidence, model serialization, and deployment handoff.

**Execution note:** Completed outputs and run IDs are preserved. Avoid retraining or re-logging completed runs unless a fresh experiment is explicitly required.

## Prerequisites & Artifact Reuse


**Execution note:** This notebook records the completed MLflow work. For a fresh runtime, the serialized Sales/Customer Random Forest and LSTM artifacts must be available before running artifact/inference cells. The project does not need to retrain models merely to display or document the completed MLflow results.


## Standalone Prerequisites & Artifact Reuse

This notebook recreates the required data variables and loads the already-trained Sales Random Forest from the project's `models` folder.

When reproducing final inference, the serialized Customer model can also be reused rather than retrained.

Expected project files when run from `notebooks/`:
- `../train.csv`
- `../store.csv`
- `../test.csv`
- `../sample_submission.csv`
- `../models/rossmann_random_forest_20260909_084440.joblib`
- `../models/rossmann_customer_random_forest_20260909_105310.joblib`
- `../mlflow.db`

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np

train = pd.read_csv('../train.csv')
store = pd.read_csv('../store.csv')
test = pd.read_csv('../test.csv')
sample_submission = pd.read_csv('../sample_submission.csv')

train['Date'] = pd.to_datetime(train['Date'])
test['Date'] = pd.to_datetime(test['Date'])
train['StateHoliday'] = train['StateHoliday'].astype(str)
test['StateHoliday'] = test['StateHoliday'].astype(str)

df = train.merge(store, on='Store', how='left')

for data in [df, test]:
    data['Year'] = data['Date'].dt.year
    data['Month'] = data['Date'].dt.month
    data['Day'] = data['Date'].dt.day
    data['WeekOfYear'] = data['Date'].dt.isocalendar().week.astype(int)
    data['IsWeekend'] = data['DayOfWeek'].isin([6, 7]).astype(int)
    data['YearMonth'] = data['Date'].dt.to_period('M').astype(str)

features = [
    'Store','DayOfWeek','Open','Promo','StateHoliday','SchoolHoliday',
    'Year','Month','Day','WeekOfYear','IsWeekend','StoreType','Assortment',
    'CompetitionDistance','CompetitionOpenSinceMonth','CompetitionOpenSinceYear',
    'Promo2','Promo2SinceWeek','Promo2SinceYear','PromoInterval'
]

test_final = test.merge(store, on='Store', how='left')
test_final['StateHoliday'] = test_final['StateHoliday'].astype(str)
test_final['Year'] = test_final['Date'].dt.year
test_final['Month'] = test_final['Date'].dt.month
test_final['Day'] = test_final['Date'].dt.day
test_final['WeekOfYear'] = test_final['Date'].dt.isocalendar().week.astype(int)
test_final['IsWeekend'] = test_final['DayOfWeek'].isin([6, 7]).astype(int)
test_final['YearMonth'] = test_final['Date'].dt.to_period('M').astype(str)

X_final_test = test_final[features]
rf_pipeline = joblib.load('../models/rossmann_random_forest_20260909_084440.joblib')

print("Data and Sales model loaded successfully.")
print("Final test shape:", X_final_test.shape)
print("Sales model type:", type(rf_pipeline))

## MLflow Installation & Experiment


In [ ]:
import importlib.util

if importlib.util.find_spec("mlflow") is not None:
    print("MLflow is installed.")
else:
    print("MLflow is not installed. Install MLflow in the project virtual environment before executing this notebook.")

MLflow is not installed.


In [ ]:
# MLflow installation is intentionally not forced in this evidence notebook.
# Project environment: Python 3.13 (.venv313) with MLflow 3.16.0.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.4/267.4 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [ ]:
import mlflow

print("MLflow version:", mlflow.__version__)
print("MLflow imported successfully.")

MLflow version: 3.16.0
MLflow imported successfully.


In [ ]:
import mlflow

mlflow.set_tracking_uri('sqlite:///../mlflow.db')

# Set the MLflow experiment
experiment_name = "Rossmann_Sales_Forecasting"

mlflow.set_experiment(experiment_name)

# Get the experiment details
experiment = mlflow.get_experiment_by_name(experiment_name)

print("Experiment name:", experiment.name)
print("Experiment ID:", experiment.experiment_id)
print("Experiment created/loaded successfully.")

2026/09/09 09:56:46 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/09 09:56:46 INFO mlflow.store.db.utils: Updating database tables
2026/09/09 09:56:49 INFO mlflow.tracking.fluent: Experiment with name 'Rossmann_Sales_Forecasting' does not exist. Creating a new experiment.


Experiment name: Rossmann_Sales_Forecasting
Experiment ID: 1
Experiment created/loaded successfully.


### MLflow Tracking Summary

The project uses MLflow to record model parameters, validation metrics, model artifacts and inference results. Important completed runs include:

- Sales Random Forest validation run: `9b2b3521098845168481e66fd17cd904`
- Sales Random Forest artifact run: `b3370602857441ceaf2a022e782a4e66`
- Multivariate LSTM validation run: `97330820481049fbbf418c62cb7aedad`
- Multivariate LSTM artifact run: `5a5f185d97d64b6782d75b42930bc25b`
- Customer Random Forest run: `5ff6cfe703ec48949f88338eb90b83eb`

The Sales artifact uses the corrected MLflow/skops trusted-type configuration required to serialize the scikit-learn pipeline successfully.

### MLflow Model Registry Evidence

The validated Sales Random Forest artifact was registered as **`Rossmann_Sales_Forecasting_RF`** with multiple registry versions.

- Version 1: registered from the validated Random Forest artifact.
- Version 2: second registry release of the same validated artifact; **no retraining was performed**.
- The MLflow UI screenshot showing both versions is retained as project evidence.

## Random Forest Sales Tracking & Artifact


In [ ]:
import mlflow

with mlflow.start_run(run_name="Random_Forest_Sales_Model") as run:
    mlflow.log_param("model", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 20)
    mlflow.log_param("min_samples_leaf", 2)
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("MAE", 809.24)
    mlflow.log_metric("RMSE", 1218.28)
    mlflow.log_metric("R2", 0.8859)

    print("Run ID:", run.info.run_id)
    print("MLflow Random Forest run logged successfully.")

Run ID: 9b2b3521098845168481e66fd17cd904
MLflow Random Forest run logged successfully.


In [ ]:
import mlflow
import mlflow.sklearn

with mlflow.start_run(run_name="Random_Forest_Model_Artifact") as run:

    # Log model parameters
    mlflow.log_param("model", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 20)
    mlflow.log_param("min_samples_leaf", 2)
    mlflow.log_param("random_state", 42)

    # Log validation metrics
    mlflow.log_metric("MAE", 809.24)
    mlflow.log_metric("RMSE", 1218.28)
    mlflow.log_metric("R2", 0.8859)

    # Log the trained Random Forest pipeline
    mlflow.sklearn.log_model(
        rf_pipeline,
        name="rossmann_random_forest_model",
        skops_trusted_types=[
            "numpy.dtype",
            "sklearn.compose._column_transformer._RemainderColsList"
        ]
    )

    print("Run ID:", run.info.run_id)
    print("Random Forest model artifact logged successfully.")

Run ID: b3370602857441ceaf2a022e782a4e66
Random Forest model artifact logged successfully.


## Multivariate LSTM Tracking & Artifact


In [ ]:
import mlflow

with mlflow.start_run(run_name="Multivariate_LSTM_Sales_Model") as run:

    # Model parameters
    mlflow.log_param("model", "Multivariate LSTM")
    mlflow.log_param("sequence_length", 28)
    mlflow.log_param("input_features", 3)
    mlflow.log_param("lstm_units", 64)
    mlflow.log_param("dropout", 0.2)
    mlflow.log_param("optimizer", "Adam")
    mlflow.log_param("loss_function", "MSE")
    mlflow.log_param("batch_size", 32)
    mlflow.log_param("max_epochs", 30)
    mlflow.log_param("shuffle", False)
    mlflow.log_param("early_stopping_patience", 5)

    # Validation metrics
    mlflow.log_metric("MAE", 504875.09)
    mlflow.log_metric("RMSE", 669092.44)
    mlflow.log_metric("R2", 0.9451)

    print("Run ID:", run.info.run_id)
    print("Multivariate LSTM parameters and metrics logged successfully.")

Run ID: 97330820481049fbbf418c62cb7aedad
Multivariate LSTM parameters and metrics logged successfully.


In [ ]:
import mlflow

with mlflow.start_run(run_name="Multivariate_LSTM_Model_Artifact") as run:

    # Log the trained LSTM model
    mlflow.log_artifact(
        "../models/rossmann_multivariate_lstm_20260909_095057.keras",
        artifact_path="lstm_model"
    )

    # Log input scaler
    mlflow.log_artifact(
        "../models/rossmann_lstm_input_scaler_20260909_095057.joblib",
        artifact_path="scalers"
    )

    # Log target scaler
    mlflow.log_artifact(
        "../models/rossmann_lstm_sales_scaler_20260909_095057.joblib",
        artifact_path="scalers"
    )

    print("Run ID:", run.info.run_id)
    print("LSTM model and scalers logged successfully.")

Run ID: 5a5f185d97d64b6782d75b42930bc25b
LSTM model and scalers logged successfully.


## MLflow Sales Inference


In [ ]:
import mlflow
import mlflow.sklearn

# MLflow Run ID containing the trained Random Forest model
rf_run_id = "b3370602857441ceaf2a022e782a4e66"

# MLflow model URI
model_uri = f"runs:/{rf_run_id}/rossmann_random_forest_model"

# Load the model directly from MLflow
rf_mlflow_model = mlflow.sklearn.load_model(model_uri)

print("Random Forest model loaded successfully from MLflow.")
print("Model type:", type(rf_mlflow_model))

Random Forest model loaded successfully from MLflow.
Model type: <class 'sklearn.pipeline.Pipeline'>


In [ ]:
# Generate predictions using the model loaded directly from MLflow
mlflow_predictions = rf_mlflow_model.predict(X_final_test)

print("MLflow prediction completed successfully.")
print("Number of predictions:", len(mlflow_predictions))
print("First 10 predictions:")
print(mlflow_predictions[:10])

MLflow prediction completed successfully.
Number of predictions: 41088
First 10 predictions:
[4964.94166091 7957.84167127 8350.69918843 6556.8465258  7699.62385178
 6493.79841401 8178.52525046 7791.35663191 5560.77204572 6318.59967692]


## Customer Random Forest Model


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

# Customer target
y_customers = df['Customers']

# Use the same features as the Sales model
X_customers = df[features]

# Chronological split
train_customer = df[df['Date'] < '2015-07-01'].copy()
val_customer = df[df['Date'] >= '2015-07-01'].copy()

X_customer_train = train_customer[features]
y_customer_train = train_customer['Customers']

X_customer_val = val_customer[features]
y_customer_val = val_customer['Customers']

# Create Customer Random Forest
customer_rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

customer_rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', customer_rf_model)
])

# Train
customer_rf_pipeline.fit(
    X_customer_train,
    y_customer_train
)

# Validation prediction
customer_val_predictions = customer_rf_pipeline.predict(
    X_customer_val
)

print("Customer prediction model trained successfully.")
print("Validation predictions:", len(customer_val_predictions))

Customer prediction model trained successfully.
Validation predictions: 34565


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

customer_mae = mean_absolute_error(
    y_customer_val,
    customer_val_predictions
)

customer_rmse = np.sqrt(
    mean_squared_error(
        y_customer_val,
        customer_val_predictions
    )
)

customer_r2 = r2_score(
    y_customer_val,
    customer_val_predictions
)

print("Customer Model Validation Metrics")
print("----------------------------------")
print("MAE :", round(customer_mae, 2))
print("RMSE:", round(customer_rmse, 2))
print("R²  :", round(customer_r2, 4))

Customer Model Validation Metrics
----------------------------------
MAE : 60.23
RMSE: 91.12
R²  : 0.9562


In [ ]:
import mlflow
import mlflow.sklearn

with mlflow.start_run(run_name="Random_Forest_Customer_Model") as run:

    # Model parameters
    mlflow.log_param("model", "RandomForestRegressor")
    mlflow.log_param("target", "Customers")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 20)
    mlflow.log_param("min_samples_leaf", 2)
    mlflow.log_param("random_state", 42)

    # Validation metrics
    mlflow.log_metric("MAE", customer_mae)
    mlflow.log_metric("RMSE", customer_rmse)
    mlflow.log_metric("R2", customer_r2)

    # Log the trained customer model
    mlflow.sklearn.log_model(
        customer_rf_pipeline,
        name="rossmann_customer_random_forest_model",
        skops_trusted_types=[
            "numpy.dtype",
            "sklearn.compose._column_transformer._RemainderColsList"
        ]
    )

    print("Run ID:", run.info.run_id)
    print("Customer Random Forest model logged successfully.")

Run ID: 5ff6cfe703ec48949f88338eb90b83eb
Customer Random Forest model logged successfully.


## MLflow Customer Inference


In [ ]:
import mlflow
import mlflow.sklearn

customer_run_id = "5ff6cfe703ec48949f88338eb90b83eb"

customer_model_uri = (
    f"runs:/{customer_run_id}/rossmann_customer_random_forest_model"
)

customer_mlflow_model = mlflow.sklearn.load_model(
    customer_model_uri
)

print("Customer Random Forest model loaded successfully from MLflow.")
print("Model type:", type(customer_mlflow_model))

Customer Random Forest model loaded successfully from MLflow.
Model type: <class 'sklearn.pipeline.Pipeline'>


In [ ]:
# Generate customer predictions using the model loaded from MLflow
mlflow_customer_predictions = customer_mlflow_model.predict(
    X_final_test
)

# Customer counts cannot be negative
mlflow_customer_predictions = mlflow_customer_predictions.clip(min=0)

print("MLflow customer prediction completed successfully.")
print("Number of predictions:", len(mlflow_customer_predictions))
print("First 10 customer predictions:")
print(mlflow_customer_predictions[:10])

MLflow customer prediction completed successfully.
Number of predictions: 41088
First 10 customer predictions:
[ 529.21721786  807.3316386   995.44481484  792.1564366   643.40436237
  631.88376033 1106.86404884  944.91362259  392.40783577  713.56219285]


## Combined Final Predictions


In [ ]:
import pandas as pd

# Create final prediction dataframe
final_predictions = pd.DataFrame({
    'Id': test_final['Id'].values,
    'Predicted_Sales': mlflow_predictions,
    'Predicted_Customers': mlflow_customer_predictions
})

print("Final prediction dataframe created successfully.")
print("Shape:", final_predictions.shape)
print("\nColumns:")
print(final_predictions.columns.tolist())

print("\nFirst 10 predictions:")
display(final_predictions.head(10))

Final prediction dataframe created successfully.
Shape: (41088, 3)

Columns:
['Id', 'Predicted_Sales', 'Predicted_Customers']

First 10 predictions:


,Id,Predicted_Sales,Predicted_Customers
0,1,4964.941661,529.217218
1,2,7957.841671,807.331639
2,3,8350.699188,995.444815
3,4,6556.846526,792.156437
4,5,7699.623852,643.404362
5,6,6493.798414,631.883760
6,7,8178.525250,1106.864049
7,8,7791.356632,944.913623
8,9,5560.772046,392.407836
9,10,6318.599677,713.562193


In [ ]:
# Save final combined predictions
final_prediction_file = "../rossmann_final_predictions.csv"

final_predictions.to_csv(
    final_prediction_file,
    index=False
)

print("Final prediction CSV saved successfully.")
print("File:", final_prediction_file)
print("Rows:", len(final_predictions))
print("Columns:", final_predictions.columns.tolist())

Final prediction CSV saved successfully.
File: /content/rossmann_final_predictions.csv
Rows: 41088
Columns: ['Id', 'Predicted_Sales', 'Predicted_Customers']


## Customer Model Serialization


In [ ]:
import joblib
from datetime import datetime
import os

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

customer_model_file = (
    f"../models/"
    f"rossmann_customer_random_forest_{timestamp}.joblib"
)

joblib.dump(
    customer_rf_pipeline,
    customer_model_file
)

print("Customer Random Forest model saved successfully.")
print("File:", customer_model_file)
print("Timestamp:", timestamp)

Customer Random Forest model saved successfully.
File: /content/rossmann_models/rossmann_customer_random_forest_20260909_105310.joblib
Timestamp: 20260909_105310


## Known Historical Validation Record


In [ ]:
# Select one known July 2015 validation record
validation_row = test_ml.iloc[0]

print("KNOWN VALIDATION RECORD")
print("-----------------------")

for feature in features:
    print(f"{feature}: {validation_row[feature]}")

print("\nActual Sales:", validation_row["Sales"])
print("Actual Customers:", validation_row["Customers"])

# Generate the model predictions for this exact record
single_input = validation_row[features].to_frame().T

sales_prediction_check = rf_pipeline.predict(single_input)[0]
customer_prediction_check = customer_rf_pipeline.predict(single_input)[0]

print("\nMODEL PREDICTIONS")
print("----------------")
print("Predicted Sales:", sales_prediction_check)
print("Predicted Customers:", customer_prediction_check)

KNOWN VALIDATION RECORD
-----------------------
Store: 1
DayOfWeek: 5
Open: 1
Promo: 1
StateHoliday: 0
SchoolHoliday: 1
Year: 2015
Month: 7
Day: 31
WeekOfYear: 31
IsWeekend: 0
StoreType: c
Assortment: a
CompetitionDistance: 1270.0
CompetitionOpenSinceMonth: 9.0
CompetitionOpenSinceYear: 2008.0
Promo2: 0
Promo2SinceWeek: nan
Promo2SinceYear: nan
PromoInterval: nan

Actual Sales: 5263
Actual Customers: 555

MODEL PREDICTIONS
----------------
Predicted Sales: 5663.062087191588
Predicted Customers: 573.6278798048344


## Final Deployment Handoff

The final prediction workflow is connected to a separate Streamlit application (`app.py`). The application loads the timestamped Sales and Customer Random Forest pipelines, accepts store/business conditions, derives calendar features, handles structural Promo2 missing values, predicts Sales and Customers, and provides a CSV download.

### Historical deployment validation

Store 1 on **2015-07-31** was used as a known validation case. The working saved Sales model and Streamlit application produced approximately **5,707.23 Sales** and **573.63 Customers**, and the CSV download reproduced those values.

### Submission artifacts

- Final combined prediction CSV: `rossmann_final_predictions.csv`
- Sales submission CSV: `rossmann_sales_predictions.csv`
- Timestamped Sales Random Forest model
- Timestamped Customer Random Forest model
- Timestamped multivariate LSTM model and scalers
- Streamlit application
- MLflow experiment and logged artifacts

## Final Submission Evidence Checklist

Completed project evidence includes MLflow runs and artifacts, MLflow inference, Customer prediction, combined predictions for 41,088 test rows, DVC dataset version history, multiple MLflow model registry versions, Streamlit application with CSV download, Power BI dashboard, and the GitHub repository.

A public hosted Streamlit URL is separate from this notebook and must be recorded if deployment is completed.

# MLflow, Final Prediction and Deployment Handoff

## Experiment tracking
The `Rossmann_Sales_Forecasting` MLflow experiment records model parameters, metrics and artifacts for reproducibility.

## Final prediction
The final prediction workflow generates Sales and Customer predictions for all 41,088 test rows and combines them with `Id`.

## Deployment
The trained Random Forest models are serialized and consumed by the Streamlit application. The application accepts store/business conditions, derives calendar features, predicts Sales and Customers, and provides a CSV download.

## Deployment validation
The historical Store 1 / 2015-07-31 test case was successfully reproduced with the saved Sales model used by Streamlit. The downloaded CSV contained the corresponding Sales and Customer predictions.
